# Level3 AI Tournament Analysis

This notebook analyzes the results of the Level3 AI tournament against Level2 and Level1 AI players.

## Analysis Includes:
- Overall win rates and performance
- Trump calling success rates
- Trick winning patterns
- Game length analysis
- Strategic behavior insights

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import numpy as np

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Load tournament results
results_files = list(Path('.').glob('tournament_results_*.json'))
if results_files:
    latest_file = max(results_files, key=lambda f: f.stat().st_mtime)
    print(f"📊 Loading results from: {latest_file}")
    
    with open(latest_file, 'r') as f:
        data = json.load(f)
    
    # Convert to DataFrames
    df_games = pd.DataFrame(data['game_results'])
    df_stats = pd.DataFrame([data['tournament_stats']])
    
    print(f"✅ Loaded {len(df_games)} games")
    print(f"📈 Tournament stats available")
else:
    print("❌ No tournament results found. Run the tournament first!")
    df_games = pd.DataFrame()
    df_stats = pd.DataFrame()

In [ ]:
# Display basic tournament information
if not df_games.empty:
    print("🏆 TOURNAMENT OVERVIEW")
    print("=" * 40)
    print(f"Total Games: {len(df_games)}")
    print(f"Date Range: {df_games['timestamp'].min()} to {df_games['timestamp'].max()}")
    print(f"Average Game Duration: {df_games['duration_seconds'].mean():.2f} seconds")
    
    # Win rates
    win_counts = df_games['winner_team'].value_counts()
    print(f"\n🏅 WIN RATES:")
    for team, count in win_counts.items():
        rate = count / len(df_games) * 100
        print(f"  {team}: {count} wins ({rate:.1f}%)")
    
    # Display first few games
    print(f"\n📋 SAMPLE GAMES:")
    display(df_games[['game_id', 'winner_team', 'final_score', 'trump_caller', 'game_length_tricks']].head())

In [ ]:
# Trump calling analysis
if not df_games.empty:
    print("🎯 TRUMP CALLING ANALYSIS")
    print("=" * 40)
    
    # Trump calling frequency
    trump_calls = df_games[df_games['trump_caller'].notna()]
    print(f"Total Trump Calls: {len(trump_calls)}")
    print(f"Trump Calling Rate: {len(trump_calls) / len(df_games) * 100:.1f}%")
    
    # Trump calling success by team
    if len(trump_calls) > 0:
        trump_success = trump_calls.groupby('trump_caller_team').agg({
            'game_id': 'count',
            'winner_team': lambda x: (x == x.index).sum()
        }).rename(columns={'game_id': 'calls', 'winner_team': 'successes'})
        
        trump_success['success_rate'] = trump_success['successes'] / trump_success['calls'] * 100
        
        print(f"\n📊 Trump Calling Success:")
        display(trump_success)
        
        # Visualize trump calling success
        plt.figure(figsize=(10, 6))
        trump_success['success_rate'].plot(kind='bar', color=['#ff6b6b', '#4ecdc4'])
        plt.title('Trump Calling Success Rate by Team')
        plt.ylabel('Success Rate (%)')
        plt.xlabel('Team')
        plt.xticks(rotation=0)
        plt.ylim(0, 100)
        
        # Add value labels on bars
        for i, v in enumerate(trump_success['success_rate']):
            plt.text(i, v + 1, f'{v:.1f}%', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Performance trends over time
if not df_games.empty:
    print("📈 PERFORMANCE TRENDS OVER TIME")
    print("=" * 40)
    
    # Convert timestamp to datetime
    df_games['timestamp_dt'] = pd.to_datetime(df_games['timestamp'])
    df_games = df_games.sort_values('timestamp_dt')
    
    # Calculate cumulative win rates
    df_games['cumulative_games'] = range(1, len(df_games) + 1)
    df_games['team_a_cumulative_wins'] = (df_games['winner_team'] == 'Team_A').cumsum()
    df_games['team_b_cumulative_wins'] = (df_games['winner_team'] == 'Team_B').cumsum()
    
    df_games['team_a_win_rate'] = df_games['team_a_cumulative_wins'] / df_games['cumulative_games']
    df_games['team_b_win_rate'] = df_games['team_b_cumulative_wins'] / df_games['cumulative_games']
    
    # Plot cumulative win rates
    plt.figure(figsize=(15, 6))
    
    plt.subplot(1, 2, 1)
    plt.plot(df_games['cumulative_games'], df_games['team_a_win_rate'], label='Team A (Level3)', color='#ff6b6b', linewidth=2)
    plt.plot(df_games['cumulative_games'], df_games['team_b_win_rate'], label='Team B (Level2+Level1)', color='#4ecdc4', linewidth=2)
    plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% Win Rate')
    plt.xlabel('Games Played')
    plt.ylabel('Cumulative Win Rate')
    plt.title('Cumulative Win Rate Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(df_games['cumulative_games'], df_games['game_length_tricks'], color='#6c5ce7', alpha=0.7)
    plt.axhline(y=df_games['game_length_tricks'].mean(), color='red', linestyle='--', alpha=0.7, label=f'Mean: {df_games["game_length_tricks"].mean():.1f}')
    plt.xlabel('Games Played')
    plt.ylabel('Game Length (tricks)')
    plt.title('Game Length Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()